# 03 - Support Constraints and L2 Regularization

In Notebook 02, we reconstructed a scene by solving a nonnegative least-squares problem using the padded linear convolution model:

$$ b = A x + n $$

where `A` pads the scene, convolves with the PSF by FFT, and crops the sensor-sized measurement.

Now we add two important priors:

1. **Support constraint**: the object is only allowed to exist in a known region.
2. **L2 regularization**: large-energy reconstructions are discouraged.


## Setup


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
from PIL import Image

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'diffusercam_sim').exists():
            return candidate
    raise RuntimeError('Could not find project root containing src/diffusercam_sim')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from diffusercam_sim import (
    PaddedLinearConvolution,
    add_gaussian_noise,
    center_support_mask,
    fista,
    make_diffuser_like_psf,
    make_synthetic_scene,
    psnr,
    residual_relative_l2,
    ssim,
)
from diffusercam_sim.viz import save_convergence_plot, save_montage

RESULT_DIR = PROJECT_ROOT / 'results' / 'notebooks' / '03_support_constraints_l2_regularization'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## The Baseline Optimization Problem

Notebook 02 solved:

$$ \min_x \frac{1}{2}\|Ax-b\|_2^2 \quad \text{subject to } x \ge 0 $$

The data term says: choose an image `x` whose simulated sensor measurement `Ax` matches the actual measurement `b`.

The nonnegativity constraint says: scene intensity cannot be negative.

The gradient of the data term is:

$$ \nabla \left(\frac{1}{2}\|Ax-b\|_2^2\right) = A^H(Ax-b) $$

This is why the adjoint operator matters. The residual `Ax-b` lives on the sensor, and `A^H` backprojects that residual into the scene estimate.

## Support Constraint

In the padded model, the optimization variable lives on a larger grid than the displayed image. Without a support constraint, the solver may place energy outside the region we intend to call the object.

Let `M` be a binary mask with value `1` where the object is allowed and `0` elsewhere. The support-constrained feasible set is:

$$ \mathcal{C} = \{x : x \ge 0 \; \text{and} \; x = M \odot x\} $$

The projection onto this set is simple:

$$ P_\mathcal{C}(z) = M \odot \max(z, 0) $$

Conceptually, this says: first remove negative intensities, then zero out everything outside the allowed support.

## L2 Regularization

L2 regularization adds a penalty on total reconstruction energy:

$$ \min_x \frac{1}{2}\|Ax-b\|_2^2 + \frac{\lambda}{2}\|x\|_2^2 $$

with the same optional constraints as before.

The derivative of the L2 term is:

$$ \nabla \left(\frac{\lambda}{2}\|x\|_2^2\right) = \lambda x $$

so the full gradient becomes:

$$ \nabla f(x) = A^H(Ax-b) + \lambda x $$

L2 regularization usually reduces noise amplification and overfitting, but it can also reduce contrast and fine detail if `lambda` is too large.

## FISTA With Projection

FISTA evaluates the gradient at a momentum point `y_k`, then projects the result:

$$ x_{k+1} = P_\mathcal{C}\left(y_k - \alpha \left[A^H(Ay_k-b) + \lambda y_k\right]\right) $$

where `alpha` is the step size. A safe step size is based on the Lipschitz constant:

$$ L \approx \|A\|_2^2 + \lambda $$

and we choose `alpha` below the stability limit. Our solver computes this from the FFT transfer function.

## Build the Forward Problem

We use a slightly noisier measurement than Notebook 02. At very high SNR, L2 regularization mostly smooths away real detail; at moderate SNR, it demonstrates the noise/detail tradeoff more clearly.

In [ ]:
image_size = 256
snr_db = 25.0

scene = make_synthetic_scene(image_size)
psf = make_diffuser_like_psf(image_size)
operator = PaddedLinearConvolution(psf)

clean_measurement = operator.forward(scene)
measurement, noise_sigma = add_gaussian_noise(clean_measurement, snr_db)
support_mask = center_support_mask(operator)

adjoint_error = operator.adjoint_inner_product_error()
adjoint_error

In [ ]:
save_montage(
    [
        ('Ground truth scene', scene),
        ('PSF, log display', psf),
        ('Noisy sensor measurement', measurement),
    ],
    RESULT_DIR / 'forward_problem.png',
    columns=3,
)
Image.open(RESULT_DIR / 'forward_problem.png')

## Run Four Reconstructions

We compare four cases:

1. **Baseline**: nonnegativity only.
2. **Support**: nonnegativity plus known object support.
3. **L2**: nonnegativity plus L2 regularization.
4. **Support + L2**: both priors together.

The point is not that one setting wins forever. The point is to see what each prior does.

In [ ]:
iterations = 80
record_every = 5
l2_lambda = 3e-4

experiments = [
    {
        'name': 'baseline',
        'title': 'Baseline',
        'support_mask': None,
        'l2_regularization': 0.0,
    },
    {
        'name': 'support',
        'title': 'Support',
        'support_mask': support_mask,
        'l2_regularization': 0.0,
    },
    {
        'name': 'l2',
        'title': 'L2',
        'support_mask': None,
        'l2_regularization': l2_lambda,
    },
    {
        'name': 'support_l2',
        'title': 'Support + L2',
        'support_mask': support_mask,
        'l2_regularization': l2_lambda,
    },
]

results = {}
for experiment in experiments:
    result = fista(
        operator=operator,
        measurement=measurement,
        iterations=iterations,
        truth=scene,
        record_every=record_every,
        support_mask=experiment['support_mask'],
        l2_regularization=experiment['l2_regularization'],
    )
    results[experiment['name']] = result

[(name, result.history[-1]) for name, result in results.items()]

## Quantitative Comparison

A lower residual means the reconstruction matches the measurement better. Higher PSNR/SSIM means it matches the known ground truth better.

In real experiments we do not know the ground truth, so residuals and visual sanity checks matter. In simulation we can compute all of them.

In [ ]:
summary = {}
for experiment in experiments:
    name = experiment['name']
    result = results[name]
    reconstruction = np.clip(result.reconstruction, 0.0, 1.0)
    predicted = operator.forward_padded(result.padded_estimate)
    summary[name] = {
        'l2_regularization': experiment['l2_regularization'],
        'uses_support_constraint': experiment['support_mask'] is not None,
        'objective': result.history[-1].objective,
        'relative_residual_l2': residual_relative_l2(measurement, predicted),
        'psnr_db': psnr(scene, reconstruction),
        'ssim': ssim(scene, reconstruction),
        'step_size': result.step_size,
    }

print(json.dumps(summary, indent=2))

## Visual Comparison

The support constraint often improves the displayed image because it prevents the solver from hiding energy outside the intended object region. L2 regularization usually makes the image smoother and less noisy, but it can reduce contrast if too strong.

In [ ]:
save_montage(
    [
        ('Ground truth scene', scene),
        ('Measurement', measurement),
        ('Baseline', results['baseline'].reconstruction),
        ('Support', results['support'].reconstruction),
        ('L2', results['l2'].reconstruction),
        ('Support + L2', results['support_l2'].reconstruction),
    ],
    RESULT_DIR / 'reconstruction_comparison.png',
    columns=3,
)
Image.open(RESULT_DIR / 'reconstruction_comparison.png')

## Convergence Comparison

The objectives are not directly comparable when L2 is enabled because the objective includes an extra penalty term. PSNR is easier to compare in simulation because ground truth is known.

In [ ]:
save_convergence_plot(
    [
        ('baseline', results['baseline'].history),
        ('support', results['support'].history),
        ('l2', results['l2'].history),
        ('support+l2', results['support_l2'].history),
    ],
    RESULT_DIR / 'convergence.png',
)
Image.open(RESULT_DIR / 'convergence.png')

## Sweep the L2 Strength

Regularization strength is not universal. It depends on the PSF, noise level, exposure, normalization, and scene class.

A small sweep is the honest way to see the tradeoff.

In [ ]:
l2_values = [0.0, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3]
sweep = []

for value in l2_values:
    result = fista(
        operator=operator,
        measurement=measurement,
        iterations=iterations,
        truth=scene,
        record_every=iterations,
        support_mask=support_mask,
        l2_regularization=value,
    )
    reconstruction = np.clip(result.reconstruction, 0.0, 1.0)
    sweep.append(
        {
            'lambda': value,
            'psnr_db': psnr(scene, reconstruction),
            'ssim': ssim(scene, reconstruction),
            'relative_residual_l2': result.history[-1].relative_residual_l2,
        }
    )

print(json.dumps(sweep, indent=2))

## Interpretation

Important patterns to look for:

- **Support may increase residual but improve image quality.** This happens because the support constraint prevents the solver from using physically impossible regions to fit noise or boundary artifacts.
- **L2 can improve robustness but blur contrast.** Larger `lambda` penalizes energy, so it suppresses noisy high-amplitude artifacts, but it also pulls real structures downward.
- **Simulation lets us tune with ground truth.** On hardware, we will not have PSNR/SSIM for real scenes, so these experiments teach us how residuals, visual artifacts, and parameter choices relate before we build the camera.

Next step: add total variation regularization. That will push us toward ADMM or proximal algorithms because TV is not as simple as adding `lambda*x` to the gradient.